<a href="https://colab.research.google.com/github/eltioxkypper/curso_ai_llama/blob/main/Challenge_Ajuste_de_Tono_con_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 53.5 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Modelo base seleccionado para el challenge
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Configuración de cuantización en 4 bits (BitsAndBytes)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

# Cargar tokenizador
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Cargar modelo base
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Preparar modelo para entrenamiento k-bit
model = prepare_model_for_kbit_training(model)
print("¡Modelo base y tokenizador cargados con éxito!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

¡Modelo base y tokenizador cargados con éxito!


In [3]:
from peft import LoraConfig, TaskType

# Definición de la configuración de LoRA requerida por el challenge
lora_config = LoraConfig(
    r=16,                           # Rango de las matrices de baja dimensión
    lora_alpha=32,                  # Parámetro de escala alfa
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Capas objetivo comunes en LLMs modernos
    lora_dropout=0.05,              # Dropout para prevenir sobreajuste
    bias="none",
    task_type=TaskType.CAUSAL_LM    # Tarea de modelado causal de lenguaje
)

# Aplicar los adaptadores LoRA sobre el modelo base
model = get_peft_model(model, lora_config)

# Imprimir parámetros entrenables para verificar
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [4]:
from datasets import Dataset

# Definir un dataset propio de pares entrada/respuesta (ej. instrucciones académicas o formato de asistencia)
raw_data = {
    "instruction": [
        "Explica qué es la fragmentación horizontal en bases de datos distribuidas.",
        "¿Cuál es la diferencia principal entre LoraConfig y un fine-tuning completo?",
        "Mide la importancia de los índices en bases de datos relacionales grandes.",
        "¿Cómo se define una clave candidata en fragmentación vertical?"
    ],
    "response": [
        "La fragmentación horizontal consiste en dividir una relación en subconjuntos de tuplas (filas) asignadas a diferentes sitios según una condición lógica.",
        "Un fine-tuning completo actualiza todos los pesos del modelo (alto costo computacional), mientras que LoRA congela el modelo base y solo entrena matrices de adaptación de bajo rango.",
        "Los índices permiten acelerar significativamente las consultas al evitar recorridos secuenciales completos sobre tablas masivas, optimizando el rendimiento de recuperación.",
        "Se define como un conjunto mínimo de atributos que identifica de manera única cada tupla, asegurando que cualquier fragmentación vertical permita un join sin pérdida de información."
    ]
}

dataset = Dataset.from_dict(raw_data)

# Función de formato con plantilla de chat/instrucción
def format_prompts(batch):
    formatted_texts = []
    for inst, resp in zip(batch["instruction"], batch["response"]):
        text = f"### Instrucción:\n{inst}\n\n### Respuesta:\n{resp}{tokenizer.eos_token}"
        formatted_texts.append(text)
    return {"text": formatted_texts}

formatted_dataset = dataset.map(format_prompts, batched=True)

# Tokenización del dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)
print("¡Dataset propio construido y tokenizado correctamente!")

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

¡Dataset propio construido y tokenizado correctamente!


In [5]:
import math

# Evaluación rápida de la pérdida base con un ciclo de evaluación sencillo
model.eval()
eval_loss = 0.0
with torch.no_grad():
    # Tomamos el primer ejemplo tokenizado como muestra de evaluación
    input_ids = torch.tensor([tokenized_dataset[0]["input_ids"]]).to(model.device)
    labels = input_ids.clone()

    outputs = model(input_ids=input_ids, labels=labels)
    initial_loss = outputs.loss.item()

print(f"📊 Métrica Cuantitativa Inicial (Loss antes de LoRA): {initial_loss:.4f}")

📊 Métrica Cuantitativa Inicial (Loss antes de LoRA): 11.4865


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./resultados_lora_challenge",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    num_train_epochs=3,               # Épocas de entrenamiento para asegurar convergencia
    max_steps=15,                     # Pasos acotados para ejecución rápida en Colab
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

# Ejecutar el entrenamiento con Fine-Tuning LoRA
print("🚀 Iniciando el proceso de Fine-tuning con LoRA...")
trainer_stats = trainer.train()
print("¡Entrenamiento finalizado con éxito!")

🚀 Iniciando el proceso de Fine-tuning con LoRA...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


In [1]:
# Evaluación de la pérdida final después del entrenamiento
model.eval()
with torch.no_grad():
    input_ids = torch.tensor([tokenized_dataset[0]["input_ids"]]).to(model.device)
    labels = input_ids.clone()

    outputs = model(input_ids=input_ids, labels=labels)
    final_loss = outputs.loss.item()

print("="*50)
print("📈 REPORTE DE EVALUACIÓN CUANTITATIVA DEL AJUSTE")
print("="*50)
print(f"• Pérdida (Loss) Inicial (Antes del entrenamiento): {initial_loss:.4f}")
print(f"• Pérdida (Loss) Final   (Después del entrenamiento): {final_loss:.4f}")
print(f"• Reducción de la Pérdida (Mejora): {initial_loss - final_loss:.4f}")
print("="*50)

NameError: name 'model' is not defined

In [ ]:
prompt = "### Instrucción:\nExplica qué es la fragmentación horizontal en bases de datos distribuidas.\n\n### Respuesta:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
model.eval()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)